# Kaggle: Làm dày dữ liệu tương tác (Dense Interactions Enrichment) cho 152.086 Items

## 1. Mục tiêu Notebook
- **Giữ nguyên 100% danh mục 152.086 sản phẩm** (`items.parquet`) để **KHÔNG CẦN** trích xuất lại các vector CLIP (Ảnh 1024-d, Text 1024-d).
- Nới lỏng bộ lọc `helpful_vote` từ $\ge 3$ xuống $\ge 0$ (hoặc $\ge 1$), thu thập toàn bộ các giao dịch mua hàng đã xác thực (`verified_purchase = True`).
- Hàn gắn lại đồ thị đồng xuất hiện (co-occurrence graph), nâng tần suất tương tác của các sản phẩm phổ biến từ 1 lên 10, 50, 100+ lượt mua.
- Thực hiện chia tách thời gian chuẩn xác (Chronological Leave-Last-Out: Train, Valid, Test) và đóng gói thành file ZIP để tải về máy cục bộ.

In [ ]:
%pip install -q 'duckdb>=1.1,<2' orjson pyarrow requests psutil

In [ ]:
import os, sys, gc, gzip, hashlib, io, json, shutil, subprocess, psutil, zipfile
from pathlib import Path
from collections import Counter
import duckdb, orjson, requests
import pyarrow as pa, pyarrow.parquet as pq

# ==========================================================================
# CẤU HÌNH THAM SỐ (HYPERPARAMETERS)
# ==========================================================================
TARGET_USERS = 35_000          # Số lượng user mục tiêu (~30k - 35k)
TARGET_INTERACTIONS = 400_000   # Giới hạn tương tác (~400k dòng, tăng gấp đôi)
MIN_USER_DEGREE = 5            # Mỗi user tối thiểu 5 tương tác (>=3 train, 1 val, 1 test)
MIN_ITEM_DEGREE = 1            # Đảm bảo toàn bộ 152k item đều có cơ hội xuất hiện
POSITIVE_RATING = 4.0          # Rating >= 4.0 tính là positive
MIN_RATING = 1.0               # Lấy toàn dải rating
ONLY_VERIFIED_PURCHASE = True  # Chỉ lấy giao dịch mua hàng đã xác thực
MIN_HELPFUL_VOTES = 0          # <-- ĐIỀU CHỈNH then chốt: Nới lỏng về 0 để không bẻ gãy đồ thị
SEED = '20260813'
CANDIDATE_HASH_MODULUS = 8     # 1/8 user pool (mở rộng từ 1/32 để lấy thêm user cho 152k item)

WORK = Path('/kaggle/working/dense_stream_subset')
WORK.mkdir(parents=True, exist_ok=True)
REVIEW_URL = 'https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Clothing_Shoes_and_Jewelry.jsonl.gz'

vm = psutil.virtual_memory()
available_gb = vm.available / 1024**3
duckdb_gb = min(24.0, max(0.5, (available_gb - 4.0) * 0.85))
print(f'Available RAM: {available_gb:.2f} GiB | DuckDB ceiling: {duckdb_gb:.2f} GiB')

## 2. Nạp danh mục Whitelist 152.086 Items hiện tại

Tìm file `items.parquet` trong thư mục `/kaggle/input` (bạn upload file `items.parquet` 49MB lên Kaggle Dataset).

In [ ]:
items_path = None
for p in Path('/kaggle/input').rglob('items.parquet'):
    items_path = p
    break

if items_path is None:
    # Fallback nếu người dùng đặt ở thư mục hiện tại
    if Path('items.parquet').exists():
        items_path = Path('items.parquet')
    else:
        raise FileNotFoundError(
            'Không tìm thấy items.parquet! Hãy upload file data/items.parquet (49MB) lên Kaggle Dataset '
            'và thêm vào Input của Notebook này.'
        )

print(f'Đang nạp danh mục item từ: {items_path}')
target_items_table = pq.read_table(items_path, columns=['item_id'])
target_items_set = set(target_items_table['item_id'].to_pylist())
print(f'Số lượng sản phẩm trong Whitelist: {len(target_items_set):,} items')
assert len(target_items_set) == 152086, f'Expected 152,086 items, got {len(target_items_set)}'

## 3. Streaming Pass 1 — Đếm số lượt tương tác hợp lệ của User trên 152k Items

In [ ]:
def stable_hash(value: str, salt: str = SEED) -> int:
    return int.from_bytes(hashlib.blake2b(f'{salt}:{value}'.encode(), digest_size=8).digest(), 'big')

def is_candidate(user_id: str) -> bool:
    return stable_hash(user_id) % CANDIDATE_HASH_MODULUS == 0

RAW_GZ_PATH = WORK / 'reviews_raw.jsonl.gz'

def ensure_raw_download():
    # Tải file gz về đĩa cục bộ MỘT LẦN để Pass 1 và Pass 2 cùng đọc lại từ đĩa,
    # tránh phải tải lại toàn bộ file qua mạng hai lần (giảm ~1 nửa thời gian chạy
    # và tránh rủi ro rớt kết nối giữa chừng ở Pass 2 sau khi đã tốn 8+ phút).
    if RAW_GZ_PATH.exists() and RAW_GZ_PATH.stat().st_size > 0:
        print(f'Đã có sẵn file tải về: {RAW_GZ_PATH} ({RAW_GZ_PATH.stat().st_size/1024**2:.1f} MiB)')
        return
    print(f'Đang tải dữ liệu thô từ: {REVIEW_URL}')
    tmp_path = RAW_GZ_PATH.with_suffix('.tmp')
    with requests.get(REVIEW_URL, stream=True, timeout=60) as response:
        response.raise_for_status()
        with open(tmp_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                if chunk:
                    f.write(chunk)
    tmp_path.rename(RAW_GZ_PATH)
    print(f'Đã tải xong: {RAW_GZ_PATH} ({RAW_GZ_PATH.stat().st_size/1024**2:.1f} MiB)')

def stream_jsonl_gz(_url: str, label: str):
    with gzip.GzipFile(str(RAW_GZ_PATH)) as gz:
        reader = io.BufferedReader(gz, buffer_size=4 * 1024 * 1024)
        count = 0
        for line in reader:
            count += 1
            if count % 2_000_000 == 0:
                print(f'[{label}] Đã quét {count:,} dòng...')
            try:
                yield orjson.loads(line)
            except Exception:
                continue

ensure_raw_download()

candidate_counts = Counter()
for row in stream_jsonl_gz(REVIEW_URL, 'reviews / pass 1'):
    user_id = row.get('user_id')
    item_id = row.get('parent_asin')
    # CHỈ LẤY sản phẩm nằm trong danh mục 152.086 items đã có
    if not user_id or item_id not in target_items_set:
        continue
    
    verified = row.get('verified_purchase') is True
    if ONLY_VERIFIED_PURCHASE and not verified:
        continue
    
    rating = float(row.get('rating') or 0)
    if rating < MIN_RATING:
        continue
    
    helpful = int(row.get('helpful_vote') or 0)
    if helpful < MIN_HELPFUL_VOTES:
        continue
    
    if is_candidate(user_id):
        candidate_counts[user_id] += 1

eligible = {u: n for u, n in candidate_counts.items() if n >= MIN_USER_DEGREE}
print(f'Candidate users: {len(candidate_counts):,}; Eligible users (>= {MIN_USER_DEGREE} reviews): {len(eligible):,}')
del candidate_counts
gc.collect()

# Phân tầng hoạt động user (Activity Strata)
def band(n):
    return '05_09' if n <= 9 else ('10_19' if n <= 19 else '20_plus')

groups = {'05_09': [], '10_19': [], '20_plus': []}
for user_id, degree in eligible.items():
    groups[band(degree)].append(user_id)

sizes = {name: len(users) for name, users in groups.items()}
total = sum(sizes.values())
floor = int(TARGET_USERS * 0.15)
quota = {name: min(sizes[name], max(floor, round(TARGET_USERS * sizes[name] / total))) for name in groups}
while sum(quota.values()) > TARGET_USERS:
    largest = max(quota, key=quota.get)
    quota[largest] -= 1

selected_users = set()
for name, users in groups.items():
    selected_users.update(sorted(users, key=lambda u: stable_hash(u, SEED + ':select'))[:quota[name]])

print('Activity strata sizes:', sizes)
print('Selected quota:', quota)
print(f'Tổng số user được chọn: {len(selected_users):,}')

## 4. Streaming Pass 2 — Ghi lại toàn bộ lịch sử của Selected Users thành Parquet

In [ ]:
raw_candidate = WORK / 'candidate_interactions.parquet'
schema = pa.schema([
    ('user_id', pa.string()),
    ('item_id', pa.string()),
    ('rating', pa.float32()),
    ('timestamp', pa.int64()),
    ('verified_purchase', pa.bool_()),
    ('is_positive', pa.int32()),
    ('review_title', pa.string()),
    ('review_text', pa.string()),
    ('helpful_vote', pa.int32())
])

writer = pq.ParquetWriter(raw_candidate, schema, compression='zstd')
batch = []
kept = 0

try:
    for row in stream_jsonl_gz(REVIEW_URL, 'reviews / pass 2'):
        user_id = row.get('user_id')
        item_id = row.get('parent_asin')
        if user_id not in selected_users or item_id not in target_items_set:
            continue
        
        verified = row.get('verified_purchase') is True
        if ONLY_VERIFIED_PURCHASE and not verified:
            continue
        
        rating = float(row.get('rating') or 0)
        if rating < MIN_RATING:
            continue
        
        helpful = int(row.get('helpful_vote') or 0)
        if helpful < MIN_HELPFUL_VOTES:
            continue
        
        batch.append({
            'user_id': user_id,
            'item_id': item_id,
            'rating': rating,
            'timestamp': int(row.get('timestamp') or 0),
            'verified_purchase': verified,
            'is_positive': 1 if rating >= POSITIVE_RATING else 0,
            'review_title': str(row.get('title') or ''),
            'review_text': str(row.get('text') or ''),
            'helpful_vote': helpful
        })
        if len(batch) >= 100_000:
            writer.write_table(pa.Table.from_pylist(batch, schema=schema))
            kept += len(batch)
            batch.clear()
    if batch:
        writer.write_table(pa.Table.from_pylist(batch, schema=schema))
        kept += len(batch)
finally:
    writer.close()

print(f'Đã ghi thành công {kept:,} dòng tương tác ({raw_candidate.stat().st_size/1024**2:.1f} MiB)')
del selected_users
gc.collect()

## 5. Xử lý trong DuckDB: Khử trùng, Capping & K-core

In [ ]:
con = duckdb.connect(str(WORK / 'processing.duckdb'))
con.execute(f"SET memory_limit='{duckdb_gb:.2f}GB'")
con.execute('SET preserve_insertion_order=false')

# 1. Khử trùng lặp (giữ review mới nhất cho mỗi cặp user-item)
con.execute(f"""CREATE OR REPLACE TABLE core AS
SELECT * FROM read_parquet('{raw_candidate.as_posix()}')
QUALIFY row_number() OVER (PARTITION BY user_id, item_id ORDER BY timestamp DESC) = 1""")

raw_count = con.execute('SELECT count(*) FROM core').fetchone()[0]
print(f'Số lượng tương tác sau khử trùng lặp: {raw_count:,} dòng')

def kcore():
    for iteration in range(20):
        before = con.execute('SELECT count(*) FROM core').fetchone()[0]
        con.execute(f"""CREATE OR REPLACE TABLE next_core AS
        WITH u AS (SELECT user_id FROM core GROUP BY 1 HAVING count(*) >= {MIN_USER_DEGREE}),
             i AS (SELECT item_id FROM core GROUP BY 1 HAVING count(*) >= {MIN_ITEM_DEGREE})
        SELECT c.* FROM core c JOIN u USING(user_id) JOIN i USING(item_id)""")
        con.execute('DROP TABLE core')
        con.execute('ALTER TABLE next_core RENAME TO core')
        after = con.execute('SELECT count(*) FROM core').fetchone()[0]
        if after == before:
            print(f'   kcore ổn định sau {iteration} vòng lặp: {after:,} dòng')
            return
    raise RuntimeError('K-core không hội tụ')

kcore()

# 2. Cắt ngân sách nếu vượt quá TARGET_INTERACTIONS
con.execute(f"""CREATE OR REPLACE TABLE capped_users AS
WITH s AS (SELECT user_id, count(*) n FROM core GROUP BY 1),
r AS (SELECT s.*, row_number() OVER (PARTITION BY CASE WHEN n<10 THEN 1 WHEN n<20 THEN 2 ELSE 3 END ORDER BY hash(user_id || '{SEED}')) br FROM s),
x AS (SELECT *, sum(n) OVER (ORDER BY br, user_id) total_n FROM r) SELECT user_id FROM x WHERE total_n <= {TARGET_INTERACTIONS}""")

con.execute('CREATE OR REPLACE TABLE core AS SELECT c.* FROM core c JOIN capped_users USING(user_id)')
kcore()

diag = con.execute('SELECT count(*) AS n_rows, count(DISTINCT user_id) AS users, count(DISTINCT item_id) AS items FROM core').fetchdf()
print('Thống kê sau K-core cuối cùng:')
print(diag)

## 6. Phân chia Thời gian (Chronological Split: Leave-Last-Out) & Xuất dữ liệu

In [ ]:
con.execute("""CREATE OR REPLACE TABLE split AS SELECT *,
CASE row_number() OVER (PARTITION BY user_id ORDER BY timestamp DESC, item_id)
  WHEN 1 THEN 'test'
  WHEN 2 THEN 'valid'
  ELSE 'train'
END split FROM core""")

# Kiểm tra tính toàn vẹn (Sanity checks)
assert con.execute("SELECT count(*) FROM (SELECT user_id FROM split GROUP BY 1 HAVING count(DISTINCT split)<>3)").fetchone()[0] == 0, 'Lỗi: Có user thiếu partition!'
assert con.execute("SELECT count(*) FROM (SELECT user_id, max(timestamp) FILTER(WHERE split='train') a, min(timestamp) FILTER(WHERE split='valid') b, min(timestamp) FILTER(WHERE split='test') c FROM split GROUP BY 1) WHERE a>b OR b>c").fetchone()[0] == 0, 'Lỗi: Rò rỉ thời gian (Temporal leakage)!'

# Xuất 3 file Parquet tương tác mới
for part in ('train', 'valid', 'test'):
    out_file = WORK / f'{part}.parquet'
    con.execute(f"""COPY (
        SELECT user_id, item_id, rating, timestamp, verified_purchase, is_positive, review_title, review_text, helpful_vote
        FROM split WHERE split='{part}' ORDER BY user_id, timestamp
    ) TO '{out_file.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)""")
    print(f'Đã xuất: {out_file.name} ({out_file.stat().st_size / 1024**2:.2f} MB)')

# Thống kê phân phối tương tác/item mới để xem độ dày co-occurrence
counts_df = con.execute('SELECT split, count(*) AS n_rows, count(DISTINCT user_id) AS users, count(DISTINCT item_id) AS items FROM split GROUP BY 1 ORDER BY 1').fetchdf()
print('\nPhân phối các tập split:')
print(counts_df)

item_deg_df = con.execute("""
WITH item_deg AS (SELECT item_id, count(*) deg FROM split WHERE split='train' GROUP BY 1)
SELECT CASE 
  WHEN deg = 1 THEN '1 review'
  WHEN deg = 2 THEN '2 reviews'
  WHEN deg BETWEEN 3 AND 5 THEN '3-5 reviews'
  WHEN deg BETWEEN 6 AND 10 THEN '6-10 reviews'
  ELSE '>10 reviews' END AS degree_group,
  count(*) as num_items, round(count(*) * 100.0 / (SELECT count(DISTINCT item_id) FROM split WHERE split='train'), 2) as pct
FROM item_deg GROUP BY 1 ORDER BY min(deg)
""").fetchdf()
print('\nPhân phối bậc của sản phẩm trong tập Train mới:')
print(item_deg_df)

def _json_default(value):
    # duckdb.fetchdf() trả về numpy scalar (int64/float64...) mà json chuẩn không serialize được
    if hasattr(value, 'item'):
        return value.item()
    return str(value)

# Lưu manifest metadata
manifest = {
    'source': 'Amazon Reviews 2023 / Clothing_Shoes_and_Jewelry',
    'strategy': 'Dense Enrichment on Fixed 152k Catalog (Relaxed helpful_votes)',
    'seed': SEED,
    'target_users': TARGET_USERS,
    'target_interactions': TARGET_INTERACTIONS,
    'counts': counts_df.to_dict('records')
}
(WORK / 'dataset_manifest.json').write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2, default=_json_default), encoding='utf-8'
)
con.close()

# Đóng gói ZIP bằng module chuẩn zipfile (không phụ thuộc lệnh `zip` của hệ thống)
zip_out = Path('/kaggle/working/dense_interactions_subset.zip')
with zipfile.ZipFile(zip_out, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=9) as zf:
    for f in (WORK / 'train.parquet', WORK / 'valid.parquet', WORK / 'test.parquet', WORK / 'dataset_manifest.json'):
        zf.write(f, arcname=f.name)

print(f'\nHOÀN THÀNH! File zip kết quả: {zip_out} ({zip_out.stat().st_size / 1024**2:.2f} MB)')

## 7. Hướng dẫn sau khi chạy xong

1. Tải file `dense_interactions_subset.zip` từ thanh điều hướng Output bên phải của Kaggle về máy.
2. Giải nén đè 3 file (`train.parquet`, `valid.parquet`, `test.parquet`) vào thư mục `data/` của dự án trên máy bạn.
3. Chạy lại lệnh huấn luyện User Tower:
   ```powershell
   # 1. CF thuần
   python -m datn.recommenders.user_tower.cli train --content --artifacts-dir data/artifacts/user_tower_cf
   python -m datn.recommenders.user_tower.cli evaluate --split test --content --artifacts-dir data/artifacts/user_tower_cf

   # 2. CF + Text
   python -m datn.recommenders.user_tower.cli train --content text --artifacts-dir data/artifacts/user_tower_text
   python -m datn.recommenders.user_tower.cli evaluate --split test --content text --artifacts-dir data/artifacts/user_tower_text

   # 3. CF + Image
   python -m datn.recommenders.user_tower.cli train --content image --artifacts-dir data/artifacts/user_tower_image
   python -m datn.recommenders.user_tower.cli evaluate --split test --content image --artifacts-dir data/artifacts/user_tower_image

   # 4. Full Multimodal
   python -m datn.recommenders.user_tower.cli train --content image text --artifacts-dir data/artifacts/user_tower
   python -m datn.recommenders.user_tower.cli evaluate --split test --content image text --artifacts-dir data/artifacts/user_tower
   ```